Here’s a clean **Markdown version** of the problem along with the **`CREATE TABLE`** and **`INSERT`** scripts so you can practice it directly:

---

# 📌 SQL Question: 615. Average Salary — Departments vs Company

## 🗂️ Table Schema

### Table: `salary`
```markdown
+----+-------------+--------+------------+
| id | employee_id | amount | pay_date   |
+----+-------------+--------+------------+
| PK | FK          | int    | date       |
+----+-------------+--------+------------+
```

### Table: `employee`
```markdown
+-------------+---------------+
| employee_id | department_id |
+-------------+---------------+
| PK          | int           |
+-------------+---------------+
```

---

## ❓ Question
Write a SQL query to compare the **average salary of each department** with the **company’s average salary** for each pay month.  
Output should show whether the department average is **higher**, **lower**, or **same** compared to the company average.

---

## 📊 Example Input

### `salary` table
```markdown
+----+-------------+--------+------------+
| id | employee_id | amount | pay_date   |
+----+-------------+--------+------------+
| 1  | 1           | 9000   | 2017-03-31 |
| 2  | 2           | 6000   | 2017-03-31 |
| 3  | 3           | 10000  | 2017-03-31 |
| 4  | 1           | 7000   | 2017-02-28 |
| 5  | 2           | 6000   | 2017-02-28 |
| 6  | 3           | 8000   | 2017-02-28 |
```

### `employee` table
```markdown
+-------------+---------------+
| employee_id | department_id |
+-------------+---------------+
| 1           | 1             |
| 2           | 2             |
| 3           | 2             |
```

---

## 📈 Expected Output
```markdown
+-----------+---------------+-------------+
| pay_month | department_id | comparison  |
+-----------+---------------+-------------+
| 2017-03   | 1             | higher      |
| 2017-03   | 2             | lower       |
| 2017-02   | 1             | same        |
| 2017-02   | 2             | same        |
```

---

## 🛠️ SQL `CREATE TABLE` Statements
```sql
CREATE TABLE salary (
    id INT PRIMARY KEY,
    employee_id INT,
    amount INT,
    pay_date DATE
);

CREATE TABLE employee (
    employee_id INT PRIMARY KEY,
    department_id INT
);
```

---

## 📥 SQL `INSERT` Statements
```sql
INSERT INTO salary (id, employee_id, amount, pay_date) VALUES
(1, 1, 9000, '2017-03-31'),
(2, 2, 6000, '2017-03-31'),
(3, 3, 10000, '2017-03-31'),
(4, 1, 7000, '2017-02-28'),
(5, 2, 6000, '2017-02-28'),
(6, 3, 8000, '2017-02-28');

INSERT INTO employee (employee_id, department_id) VALUES
(1, 1),
(2, 2),
(3, 2);
```

---

✨ This Markdown gives you a **ready-to-use schema and dataset** for practicing the query.  

Would you like me to also include the **solution query** (with `CASE` logic comparing department vs company averages) so you can run it directly after creating and inserting the tables?

In [0]:
%sql
CREATE TABLE salary_615 (
    id INT PRIMARY KEY,
    employee_id INT,
    amount INT,
    pay_date DATE
);

CREATE TABLE employee_615 (
    employee_id INT PRIMARY KEY,
    department_id INT
);
INSERT INTO salary_615 (id, employee_id, amount, pay_date) VALUES
(1, 1, 9000, '2017-03-31'),
(2, 2, 6000, '2017-03-31'),
(3, 3, 10000, '2017-03-31'),
(4, 1, 7000, '2017-02-28'),
(5, 2, 6000, '2017-02-28'),
(6, 3, 8000, '2017-02-28');

INSERT INTO employee_615 (employee_id, department_id) VALUES
(1, 1),
(2, 2),
(3, 2);


# My Thought Process

1. **Initial Idea**  
   I considered creating a single variable to store the overall average company salary. This would avoid repeating queries and allow me to reuse the variable in the `CASE` statement.

2. **Using Two CTEs**  
   Since I wanted one query, I planned to use two CTEs:
   - The first CTE calculates the **company-wide average**.  
   - The second CTE calculates the **department-wise average**.  

   While working with the data, I extracted the month and year from the date, concatenated them into a string, and realized that `LPAD` could be used to ensure the month always has two digits.

3. **Realization and Correction**  
   - The first approach (single variable) wasn’t feasible.  
   - The second approach was incomplete because the company average also needs to be calculated **per month**, not just overall.  
   - Therefore, I had to stick with the two-CTE approach.  
   - I also realized that I cannot directly use the department average in the `CASE` statement. Instead, I need to **join both CTEs** on the month-year field.  
   - I preferred a **LEFT JOIN** over an INNER JOIN, since a department might exist without employees. Our focus is department-wise, so this choice made sense.

---

### Key Learnings
- Functions: `YEAR(date_column)`, `MONTH(date_column)`  
- String handling: `CONCAT()` is comma-separated  
- Formatting: `LPAD()` helps keep month values fixed at two digits  
- Mistake: I mistakenly tried to take an average of an already averaged value (`AVG(avg_amount)`) outside the CTE. This was incorrect, and I also forgot to include a `GROUP BY` clause. The average should only be calculated once at the right level of aggregation.


In [0]:
%sql
WITH company_avg AS (
    SELECT 
        CONCAT(YEAR(s.pay_date), '-', LPAD(MONTH(s.pay_date), 2, '0')) AS pay_month,
        AVG(s.amount) AS avg_company_salary
    FROM salary_615 s
    GROUP BY CONCAT(YEAR(s.pay_date), '-', LPAD(MONTH(s.pay_date), 2, '0'))
),
dept_avg AS (
    SELECT 
        CONCAT(YEAR(s.pay_date), '-', LPAD(MONTH(s.pay_date), 2, '0')) AS pay_month,
        e.department_id,
        AVG(s.amount) AS avg_dept_salary
    FROM employee_615 e
    LEFT JOIN salary_615 s 
        ON e.employee_id = s.employee_id
    GROUP BY e.department_id, CONCAT(YEAR(s.pay_date), '-', LPAD(MONTH(s.pay_date), 2, '0'))
)
SELECT 
    d.pay_month,
    d.department_id,
    CASE 
        WHEN d.avg_dept_salary > c.avg_company_salary THEN 'higher'
        WHEN d.avg_dept_salary < c.avg_company_salary THEN 'lower'
        ELSE 'same'
    END AS comparison
FROM dept_avg d
JOIN company_avg c 
    ON d.pay_month = c.pay_month
ORDER BY d.pay_month desc, d.department_id;


#optimized

In [0]:
%sql
with cte as (
    Select avg(amount)over(partition by concat(year(pay_date),'-',lpad(month(pay_date),2,'0'))) as company_avg_amount , 
    avg(amount)over(partition by concat(year(pay_date),'-',lpad(month(pay_date),2,'0')), department_id) as avg_amount,
    concat(year(pay_date),'-',lpad(month(pay_date),2,'0') )as pay_month ,
    department_id
    from salary_615  s left join employee_615 e
    on s.employee_id = e.employee_id
)
select distinct  pay_month , department_id, case when avg_amount >  company_avg_amount then 'Higher' 
when avg_amount <  company_avg_amount then 'lower'
else 'same'
end  as comparison
  from cte 
 order by pay_month desc , department_id asc

#my mistakes 

In [0]:
  %sql
with cte as (
    Select avg(amount) as avg_company_salary
    from  salary_615
)
, cte2 as (
  select concat(
    year(s.pay_date)
    ,'-'
    ,lpad(month(s.pay_date), 2, '0')
    ) as date_m_y , department_id ,avg(amount) as avg_amount

  from employee_615 e left  join salary_615 s
  on e.employee_id = s.employee_id
  group by date_m_y , department_id
)
Select date_m_y ,department_id,
 case 
when avg_amount > (select avg_company_salary from cte) then 'higher'
when avg_amount < (select avg_company_salary from cte) then 'lower'
else 'Same'
end
 from cte2 

In [0]:
%sql
SELECT concat(
   year(s.pay_date)
          ,  -- ensures 2-digit month
           '-',
           lpad(month(s.pay_date), 2, '0')
       ) AS month_year
FROM   salary_615 s;

In [0]:
%sql
WITH cte AS (
    SELECT AVG(amount) AS avg_company_salary
    FROM salary_615
),
cte2 AS (
    SELECT 
        CONCAT(
            YEAR(s.pay_date), '-', LPAD(MONTH(s.pay_date), 2, '0')
        ) AS date_m_y,
        e.department_id,
        AVG(s.amount) AS avg_amount
    FROM employee_615 e
    LEFT JOIN salary_615 s
        ON e.employee_id = s.employee_id
    GROUP BY 
        e.department_id,
        CONCAT(YEAR(s.pay_date), '-', LPAD(MONTH(s.pay_date), 2, '0'))
)
SELECT *
FROM cte2;
